# 🔬 Tahap 1: Eksplorasi Data (EDA) & Preprocessing Dataset Multi-Label Hate Speech & Abusive Language

**Topik Penelitian:** Deteksi *Cyberbullying* dan Ujaran Kebencian Multi-Label Berbahasa Indonesia Menggunakan IndoBERT pada Platform Discord  
**Sumber Dataset:** Muhammad Okky Ibrohim & Indra Budi (2019) - *ACL Anthology (ALW3)*  

---
### Tujuan Notebook Ini:
1. Memuat dan memeriksa dataset mentah (`re_dataset.csv`).
2. Menganalisis distribusi 12 label klasifikasi (*multi-label*).
3. Menganalisis karakteristik panjang teks (*sentence length*).
4. Memuat kamus normalisasi bahasa gaul (`new_kamusalay.csv`) dan leksikon kata kasar (`abusive.csv`).
5. Menguji fungsi pembersihan dan normalisasi teks bertahap.

## 1. Import Library Dasar

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Set style visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("✅ Semua library dasar berhasil dimuat!")

## 2. Memuat Dataset Mentah (`re_dataset.csv`)

Dataset ini berisi **13.169 tweet** berbahasa Indonesia yang telah dianotasi dengan 12 label multi-label.

In [ ]:
# Path direktori dataset
DATA_DIR = Path("../data/dataset_raw")
dataset_path = DATA_DIR / "re_dataset.csv"

# Membaca dataset dengan encoding latin-1
df = pd.read_csv(dataset_path, encoding="latin-1")

print(f"Total baris data : {df.shape[0]:,} baris")
print(f"Total kolom      : {df.shape[1]} kolom")
print("\nDaftar Kolom:", df.columns.tolist())
df.head()

## 3. Pengecekan Struktur Kolom & Missing Values

In [ ]:
print("--- Informasi Struktur Kolom ---")
print(df.info())

print("\n--- Pengecekan Missing Values (Data Kosong) ---")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Tidak ada data bernilai kosong (NaN)!")

## 4. Analisis Distribusi 12 Kategori Label (*Multi-Label Distribution*)

12 kolom label meliputi:
- **Kategori Utama:** `HS` (Hate Speech), `Abusive` (Kata Kasar)
- **Target:** `HS_Individual` (Menyerang Personal), `HS_Group` (Menyerang Kelompok)
- **Kategori SARA/Fisik:** `HS_Religion`, `HS_Race`, `HS_Physical`, `HS_Gender`, `HS_Other`
- **Tingkat Keparahan:** `HS_Weak`, `HS_Moderate`, `HS_Strong`

In [ ]:
LABEL_COLUMNS = [
    "HS", "Abusive", "HS_Individual", "HS_Group", 
    "HS_Religion", "HS_Race", "HS_Physical", "HS_Gender", 
    "HS_Other", "HS_Weak", "HS_Moderate", "HS_Strong"
]

# Hitung frekuensi kemunculan (label = 1)
label_counts = df[LABEL_COLUMNS].sum().sort_values(ascending=False)
label_percentages = (label_counts / len(df) * 100).round(2)

# Tampilkan tabel rekap
summary_labels_df = pd.DataFrame({
    "Jumlah Tweet (Positif 1)": label_counts,
    "Persentase (%)": label_percentages
})
summary_labels_df

## 5. Visualisasi Distribusi Label (Bar Chart)

In [ ]:
plt.figure(figsize=(12, 6))
palette = sns.color_palette("viridis", len(label_counts))
ax = sns.barplot(x=label_counts.values, y=label_counts.index, palette=palette)

plt.title("Distribusi 12 Label Multi-Label (Dataset Ibrohim & Budi 2019)", fontsize=14, fontweight="bold")
plt.xlabel("Jumlah Kemunculan (Label = 1)", fontsize=12)
plt.ylabel("Kategori Label", fontsize=12)

# Tambahkan label angka di ujung bar
for i, v in enumerate(label_counts.values):
    ax.text(v + 100, i, f"{v:,} ({label_percentages.iloc[i]}%)", va="center", fontsize=10, fontweight="semibold")

plt.tight_layout()
plt.show()

## 6. Analisis Karakteristik Panjang Teks (*Sentence Length Analysis*)

Karakteristik teks obrolan di Discord didominasi oleh pesan-pesan singkat. Kita analisis distribusi jumlah karakter dan kata pada dataset ini.

In [ ]:
# Hitung panjang karakter dan jumlah kata
df["char_length"] = df["Tweet"].astype(str).apply(len)
df["word_count"] = df["Tweet"].astype(str).apply(lambda x: len(x.split()))

print("--- Statistik Deskriptif Panjang Teks ---")
print(df[["char_length", "word_count"]].describe().round(1))

short_text_count = (df["char_length"] <= 100).sum()
long_text_count = (df["char_length"] > 100).sum()

print(f"\nJumlah Teks Pendek (<= 100 karakter) : {short_text_count:,} ({short_text_count/len(df)*100:.1f}%)")
print(f"Jumlah Teks Panjang (> 100 karakter)  : {long_text_count:,} ({long_text_count/len(df)*100:.1f}%)")

## 7. Memuat Kamus Slang (`new_kamusalay.csv`) dan Leksikon Kasar (`abusive.csv`)

In [ ]:
kamusalay_path = DATA_DIR / "new_kamusalay.csv"
abusive_path = DATA_DIR / "abusive.csv"

# Membaca kamus alay (kolom: slang, formal)
df_kamusalay = pd.read_csv(kamusalay_path, encoding="latin-1", header=None, names=["slang", "formal"])
slang_dict = dict(zip(df_kamusalay["slang"], df_kamusalay["formal"]))

# Membaca leksikon abusive
df_abusive = pd.read_csv(abusive_path, encoding="latin-1", header=None, names=["abusive_word"])
abusive_words = set(df_abusive["abusive_word"].str.lower().tolist())

print(f"✅ Total kata slang dalam kamusalay : {len(slang_dict):,} kata")
print(f"✅ Total leksikon kata abusive       : {len(abusive_words):,} kata")

print("\nContoh pemetaan kata slang -> formal:")
for k in list(slang_dict.keys())[:7]:
    print(f"  '{k}' -> '{slang_dict[k]}'")

## 8. Fungsi Preprocessing Teks (Penanganan Mention, URL, Slang, & Karakter Berulang)

Pipeline preprocessing khusus untuk teks media sosial & percakapan Discord:
1. **Case folding**: Mengubah semua teks menjadi huruf kecil.
2. **Pembersihan URL & Mention**: Menyeragamkan format `@user` atau mention Discord `<@ID>` menjadi token `[USER]`.
3. **Normalisasi Kata Gaul (*Slang*)**: Menerjemahkan kata slang ke bentuk formal menggunakan `slang_dict`.
4. **Reduksi Huruf Berulang**: Mengatasi pemanjangan kata (misal: `toloool` -> `tolol`, `begoook` -> `begok`).
5. **Pembersihan Karakter Non-Alfanumerik**: Menghapus tanda baca berlebih dan spasi ganda.

In [ ]:
def clean_and_normalize_text(text: str, slang_mapping: dict[str, str]) -> str:
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Ganti URL menjadi token [URL]
    text = re.sub(r"https?://\S+|www\.\S+", "[URL]", text)
    
    # 3. Ganti mention Discord (<@123456789>) atau @username menjadi [USER]
    text = re.sub(r"<@!?\d+>", " [USER] ", text)
    text = re.sub(r"@\w+", " [USER] ", text)
    text = re.sub(r"\buser\b", "[USER]", text)
    
    # 4. Hapus karakter non-alfanumerik kecuali token khusus [USER] dan [URL]
    text = re.sub(r"[^a-zA-Z0-9\[\]\s]", " ", text)
    
    # 5. Reduksi karakter berulang (misal: toloool -> tolol)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    
    # 6. Normalisasi slang word per token
    words = text.split()
    normalized_words = [slang_mapping.get(w, w) for w in words]
    
    # 7. Rapikan spasi ganda
    clean_text = " ".join(normalized_words).strip()
    return clean_text

# Uji coba fungsi preprocessing pada beberapa contoh kalimat
sample_tests = [
    "RT @budi: Woy <@987654321> toloool bgt lu, kerjaan gak becuuus! https://link.com",
    "dasar gblk lu anak haram, mati aja lu sana!",
    "Anjir gw kesel bgt game ini lag parah"
]

print("--- Hasil Uji Coba Preprocessing ---")
for s in sample_tests:
    print("Asli   :", s)
    print("Bersih :", clean_and_normalize_text(s, slang_dict))
    print()

## 9. Menerapkan Preprocessing pada Seluruh Dataset

In [ ]:
print("Sedang memproses seluruh 13.169 tweet...")
df["clean_tweet"] = df["Tweet"].apply(lambda x: clean_and_normalize_text(x, slang_dict))

print("✅ Selesai! Berikut perbandingan teks asli vs teks bersih:")
df[["Tweet", "clean_tweet"]].head(5)

## 10. Menyimpan Data Bersih (*Cleaned Dataset*)

Data yang telah dibersihkan disimpan ke `data/dataset_clean.csv` untuk digunakan pada notebook pelatihan IndoBERT selanjutnya.

In [ ]:
output_clean_path = Path("../data/dataset_clean.csv")
df.to_csv(output_clean_path, index=False)

print(f"✅ Dataset bersih berhasil disimpan ke: {output_clean_path.resolve()}")
print(f"Ukuran file: {os.path.getsize(output_clean_path) / (1024*1024):.2f} MB")